In [1]:
import matplotlib.pyplot as plt
import numpy as np

from analysis.visualizations import plot_texture_variant_embeddings, plot_texture_variant_embeddings_v2, plot_texture_variant_trajectories


In [2]:
from analysis.data_utils import (
    find_case_file, load_case_array, ensure_4d_CDHW, extract_patch_from_array,
    initialize_untrained_predictor, extract_activation_vector,
    enhance_reduce_texture, enhance_multiscale_texture, adjust_intensity,
    create_texture_modified_patches, compute_embeddings_for_texture_variants,
    create_intensity_modified_patches, compute_embeddings_for_intensity_variants
)


In [3]:
# Imports and configuration
import sys
import json
import subprocess
import importlib
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from sklearn.decomposition import PCA

# try import umap-learn
try:
    import umap
except Exception:
    print('umap not found, attempting to install umap-learn...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn'])
    importlib.invalidate_caches()
    import umap

print('torch:', torch.__version__)
print('umap:', umap.__version__)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Defaults
DEFAULT_PATCH_SIZE = (32, 160, 128)  # (D,H,W)
# DEFAULT_PATCH_SIZE = (32, 512, 512)  # (D,H,W)
ROOT_DIR = Path('/nfs/erelab001/shared/Computational_Group/Naravich')
DATASETS_DIR = ROOT_DIR / 'datasets' / 'nnUNet_Datasets'
IMAGES_DIR = DATASETS_DIR / 'nnUNet_preprocessed'/ 'Dataset307_Sohee_Calcium_OCT_CrossValidation' / 'nnUNetPlans_3d_fullres'
plans_file = DATASETS_DIR / "nnUNet_preprocessed/Dataset307_Sohee_Calcium_OCT_CrossValidation/nnUNetPlans.json"
dataset_file = DATASETS_DIR / "nnUNet_preprocessed/Dataset307_Sohee_Calcium_OCT_CrossValidation/dataset.json"
with open(dataset_file) as f:
    dataset_json = json.load(f)

# Example: user will override these two lists when using the notebook
PATCH_LIST = [
    # Default examples: small patches from various locations in one volume
    ('101-019.npy', (162,220,155)),
    ('101-019.npy', (147,170,150)),
    ('101-019.npy', (170,270,135)),
    ('101-019.npy', (147,80,290)),
    ('101-019.npy', (171,242,288)),
    ('101-019.npy', (171,299,172)),
    ('101-019.npy', (171,122,132)),
    ('101-019.npy', (160,114,310)),
    ('101-019.npy', (153-16,172,337)),
    ('101-019.npy', (153-16,345,224)),
    ('101-019.npy', (149-16,348,242)),
    ('101-019.npy', (149-16,194,183)),
    ('101-019.npy', (143-16,207,199)),
    ('101-019.npy', (108-16,300,300)),
    ('101-019.npy', (108-16,176,234)),
    ('101-019.npy', (96-16,149,212)),
    ('101-019.npy', (81-16,122,215)),
    ('101-019.npy', (81-16,190,248)),
]
PRETRAINED_MODELS = [
    # LaW
    (DATASETS_DIR / "nnUNet_results/Dataset300_Lumen_and_Wall_OCT/nnUNetTrainer__nnUNetPreprocessPlans__3d_fullres/fold_all/checkpoint_best.pth", "LaW"),
    # Genesis
    (ROOT_DIR / "datasets" / "ModelGenesisOutputs/ModelGenesisNNUNetPretrainingV2_noNorm_correct_orientation/Converted_nnUNet_Genesis_OCT_Best.pt", "Genesis"),
    # CLIP
    (DATASETS_DIR / "../OpenAI-CLIP-logs/shared_projector_shared_encoder/nnunet.pt", "CLIP"),
    # No Pretrain
    (Path("None"), "No Pretrain"),
    # str(path_to_pretrained_checkpoint_or_model_folder),
]

/nfs/erelab001/shared/Computational_Group/Naravich/nnUNet/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.5.1
umap: 0.5.9.post2
Device: cuda


In [ ]:
import torch
import os

patch_idx = 2
case_name, offset = PATCH_LIST[patch_idx]
case_id = case_name.replace('.npy', '')

texture_path = f'texture_variant_results_{case_id}_patch{patch_idx}.pth'
intensity_path = f'intensity_variant_results_{case_id}_patch{patch_idx}.pth'

if os.path.exists(texture_path) and os.path.exists(intensity_path):
    print('Loading results from .pth files...')
    texture_variant_results = torch.load(texture_path)
    intensity_variant_results = torch.load(intensity_path)
else:
    print('Computing embeddings... This may take some time.')
    case_file = find_case_file(IMAGES_DIR, case_name)
    arr = load_case_array(case_file)
    arr = ensure_4d_CDHW(arr)
    single_patch, actual_offset = extract_patch_from_array(arr, offset, DEFAULT_PATCH_SIZE)

    factors = [1e-6, 0.1, 0.2, 0.25, 0.33, 0.5, 0.75, 0.8, 0.9, 1.0, 1.1, 1.2, 1.25, 1.5, 1.67, 1.75, 1.8, 1.9, 2.0]
    custom_factors = factors
    texture_variant_results = compute_embeddings_for_texture_variants(
        single_patch,
        (case_name, offset),
        PRETRAINED_MODELS,
        plans_file,
        dataset_json,
        patch_size=DEFAULT_PATCH_SIZE,
        enhancement_factors=custom_factors
    )

    custom_intensity_factors = [1e-10, 1e-3, 0.1, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0, 5.0, 1e+1, 1e+2, 1e+3]
    intensity_variant_results = compute_embeddings_for_intensity_variants(
        single_patch, 
        (case_name, offset), 
        PRETRAINED_MODELS,
        plans_file,
        dataset_json,
        intensity_factors=custom_intensity_factors
    )

    torch.save(texture_variant_results, texture_path)
    torch.save(intensity_variant_results, intensity_path)
    print('Results saved to .pth files.')


Computing embeddings... This may take some time.


Processing model 1/4: LaW
Initialized untrained network from /nfs/erelab001/shared/Computational_Group/Naravich/datasets/nnUNet_Datasets/nnUNet_preprocessed/Dataset307_Sohee_Calcium_OCT_CrossValidation/nnUNetPlans.json
Using configuration: 3d_32x160x128_b10
Architecture: dynamic_network_architectures.architectures.unet.PlainConvUNet
Input channels: 1, Output channels: 2
  Loading pretrained weights from /nfs/erelab001/shared/Computational_Group/Naravich/datasets/nnUNet_Datasets/nnUNet_results/Dataset300_Lumen_and_Wall_OCT/nnUNetTrainer__nnUNetPreprocessPlans__3d_fullres/fold_all/checkpoint_best.pth
################### Loading pretrained weights from file  /nfs/erelab001/shared/Computational_Group/Naravich/datasets/nnUNet_Datasets/nnUNet_results/Dataset300_Lumen_and_Wall_OCT/nnUNetTrainer__nnUNetPreprocessPlans__3d_fullres/fold_all/checkpoint_best.pth ###################


/nfs/erelab001/shared/Computational_Group/Naravich/nnUNet/nnunetv2/run/load_pretrained_weights.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_model = torch.load(

Processing model 2/4: Genesis
Initialized untrained network from /nfs/erelab001/shared/Computational_Group/Naravich/datasets/nnUNet_Datasets/nnUNet_preprocessed/Dataset307_Sohee_Calcium_OCT_CrossValidation/nnUNetPlans.json
Using configuration: 3d_32x160x128_b10
Architecture: dynamic_network_architectures.architectures.unet.PlainConvUNet
Input channels: 1, Output channels: 2
  Loading pretrained weights from /nfs/erelab001/shared/Computational_Group/Naravich/datasets/ModelGenesisOutputs/ModelGenesisNNUNetPretrainingV2_noNorm_correct_orientation/Converted_nnUNet_Genesis_OCT_Best.pt
################### Loading pretrained weights from file  /nfs/erelab001/shared/Computational_Group/Naravich/datasets/ModelGenesisOutputs/ModelGenesisNNUNetPretrainingV2_noNorm_correct_orientation/Converted_nnUNet_Genesis_OCT_Best.pt ###################


/nfs/erelab001/shared/Computational_Group/Naravich/nnUNet/nnunetv2/run/load_pretrained_weights.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_model = torch.load(

...

Processing model 2/4: Genesis
Initialized untrained network from /nfs/erelab001/shared/Computational_Group/Naravich/datasets/nnUNet_Datasets/nnUNet_preprocessed/Dataset307_Sohee_Calcium_OCT_CrossValidation/nnUNetPlans.json
Using configuration: 3d_32x160x128_b10
Architecture: dynamic_network_architectures.architectures.unet.PlainConvUNet
Input channels: 1, Output channels: 2
  Loading pretrained weights from /nfs/erelab001/shared/Computational_Group/Naravich/datasets/ModelGenesisOutputs/ModelGenesisNNUNetPretrainingV2_noNorm_correct_orientation/Converted_nnUNet_Genesis_OCT_Best.pt
################### Loading pretrained weights from file  /nfs/erelab001/shared/Computational_Group/Naravich/datasets/ModelGenesisOutputs/ModelGenesisNNUNetPretrainingV2_noNorm_correct_orientation/Converted_nnUNet_Genesis_OCT_Best.pt ###################


/nfs/erelab001/shared/Computational_Group/Naravich/nnUNet/nnunetv2/run/load_pretrained_weights.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_model = torch.load(

Processing model 3/4: CLIP
Initialized untrained network from /nfs/erelab001/shared/Computational_Group/Naravich/datasets/nnUNet_Datasets/nnUNet_preprocessed/Dataset307_Sohee_Calcium_OCT_CrossValidation/nnUNetPlans.json
Using configuration: 3d_32x160x128_b10
Architecture: dynamic_network_architectures.architectures.unet.PlainConvUNet
Input channels: 1, Output channels: 2
  Loading pretrained weights from /nfs/erelab001/shared/Computational_Group/Naravich/datasets/nnUNet_Datasets/../OpenAI-CLIP-logs/shared_projector_shared_encoder/nnunet.pt
################### Loading pretrained weights from file  /nfs/erelab001/shared/Computational_Group/Naravich/datasets/nnUNet_Datasets/../OpenAI-CLIP-logs/shared_projector_shared_encoder/nnunet.pt ###################


/nfs/erelab001/shared/Computational_Group/Naravich/nnUNet/nnunetv2/run/load_pretrained_weights.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_model = torch.load(

Processing model 4/4: No Pretrain
Initialized untrained network from /nfs/erelab001/shared/Computational_Group/Naravich/datasets/nnUNet_Datasets/nnUNet_preprocessed/Dataset307_Sohee_Calcium_OCT_CrossValidation/nnUNetPlans.json
Using configuration: 3d_32x160x128_b10
Architecture: dynamic_network_architectures.architectures.unet.PlainConvUNet
Input channels: 1, Output channels: 2
Results saved to .pth files.


In [ ]:
# Visualize: Texture Variants with Color-Coded Points

plot_texture_variant_embeddings(texture_variant_results, figsize=(14, 10))
plot_texture_variant_embeddings_v2(texture_variant_results, figsize=(14, 10))
plot_texture_variant_trajectories(texture_variant_results, figsize=(16, 10))


In [ ]:
# Visualize: Intensity Variants with Color-Coded Points

plot_texture_variant_embeddings(intensity_variant_results, figsize=(14, 10))
plot_texture_variant_embeddings_v2(intensity_variant_results, figsize=(14, 10))


In [ ]:
# Visualize: Embedding Trajectory (with connecting lines)

plot_texture_variant_trajectories(intensity_variant_results, figsize=(16, 10))
